In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '03_02_task_matching_and_coefficient_comparison.ipynb'
NOTEBOOK_RELATIVE_PATH = '03_02_task_matching_and_coefficient_comparison.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
import numpy as u00_np
import pandas as u00_pd
from scipy import stats as u00_stats


In [ ]:
def u00_behavior_scored_trials():
    d = u00_behavior().copy()
    d['rt_valid'] = d.correct.eq(1) & d.rt_s.notna()
    for ((sid, cond), g) in d[d.task.eq('stroop')].groupby(['subject_id', 'condition']):
        r = g.loc[g.rt_valid, 'rt_s']
        sd = r.std(ddof=1)
        if len(r) > 1:
            d.loc[g.index, 'rt_valid'] &= d.loc[g.index, 'rt_s'].between(r.mean() - 3 * sd, r.mean() + 3 * sd)
    gng = d.task.eq('gonogo')
    d.loc[gng, 'rt_valid'] &= d.loc[gng, 'rt_s'].between(0.15, 0.5)
    return d


In [ ]:
def u00_behavior_subject_scores(d=None):
    d = u00_behavior_scored_trials() if d is None else d
    rows = []
    for (sid, g) in d.groupby('subject_id'):
        st = g[g.task.eq('stroop')]
        ng = g[g.task.eq('gonogo')]
        go = ng[ng.condition.eq('go')]
        no = ng[ng.condition.eq('nogo')]
        row = {'subject_id': sid}
        for c in ['congruent', 'neutral', 'incongruent']:
            b = st[st.condition.eq(c)]
            row['stroop_' + c + '_rt_ms'] = b.loc[b.rt_valid, 'rt_s'].mean() * 1000
            row['stroop_' + c + '_accuracy'] = b.correct.mean()
        row['stroop_rt_cost_ms'] = row['stroop_incongruent_rt_ms'] - row['stroop_congruent_rt_ms']
        row['stroop_accuracy_cost_pct'] = (row['stroop_congruent_accuracy'] - row['stroop_incongruent_accuracy']) * 100
        row['stroop_mean_rt_ms'] = st.loc[st.rt_valid, 'rt_s'].mean() * 1000
        row['gonogo_go_rt_ms'] = go.loc[go.rt_valid, 'rt_s'].mean() * 1000
        row['go_accuracy'] = go.correct.mean()
        row['nogo_accuracy'] = no.correct.mean()
        row['gonogo_commission_pct'] = (1 - row['nogo_accuracy']) * 100
        row['gonogo_omission_pct'] = (1 - row['go_accuracy']) * 100
        row['gonogo_dprime'] = u00_stats.norm.ppf((go.correct.sum() + 0.5) / (len(go) + 1)) - u00_stats.norm.ppf(((1 - no.correct).sum() + 0.5) / (len(no) + 1))
        row['accuracy_difference'] = row['nogo_accuracy'] - row['go_accuracy']
        rows.append(row)
    result = u00_pd.DataFrame(rows)
    if not u00_np.isfinite(result.drop(columns='subject_id').to_numpy(float)).all():
        raise ValueError('Insufficient valid trials for a behavioral measure')
    return result


In [ ]:
def u00_behavior_analysis_data():
    d = u00_posterior().merge(u00_participants(), on='subject_id', validate='one_to_one').merge(u00_behavior_subject_scores(), on='subject_id', validate='one_to_one')
    for (task, cols) in [('stroop', {'a': 'stroop_a', 'v': 'stroop_v_congruent', 't': 'stroop_t'}), ('gonogo', {'a': 'gonogo_a', 'v': 'gonogo_v_go', 't': 'gonogo_t', 'z': 'gonogo_z'})]:
        for (short, long) in cols.items():
            d[short + '_' + task] = d[long]
    d['z_stroop'] = 0.5
    d['v_stroop_conflict'] = d.stroop_v_incongruent - d.stroop_v_congruent
    d['education_years'] = d.education
    d['data_status'] = 'OBSERVED_INPUTS'
    return d


In [ ]:
from pathlib import Path as u01_Path
from pathlib import Path as u01_Path
import json as u01_json, re as u01_re
import numpy as u01_np
import pandas as u01_pd


In [ ]:
def u01_ddm_fit_hddm(samples=15000, burn=5000, chains=4, seed=20261021, data=None, output_dir=None, model_check=False, ppc_samples=100, export_long=False):
    import hddm as hddm
    u01_ddm_validate_hddm_backend(hddm)
    if data is not None and (not model_check):
        raise ValueError('Explicit in-memory data are reserved for labelled model checks')
    d = u01_observed_hddm_data() if data is None else data.copy()
    d['subj_idx'] = d.subj_idx.astype(str)
    ids = set(u01_participants().subject_id)
    if set(d.subj_idx) != ids or set(d.task) != {'stroop', 'gonogo'}:
        raise ValueError('HDDM input cohort or task mismatch')
    if not d.response.isin([0, 1]).all() or not u01_np.isfinite(d.rt).all() or (d.rt <= 0).any():
        raise ValueError('Positive observed RT in seconds; response codes 0/1; missing RT encoded 999')
    if burn >= samples or chains < 2:
        raise ValueError('Multiple chains and samples > burn are required')
    out = u01_Path(output_dir) if output_dir is not None else u01_ddm_OUT
    (out / 'tables').mkdir(parents=True, exist_ok=True)
    fit_state = None
    if not model_check:
        import hashlib as hashlib
        fit_state = dict(status='running', samples=samples, burn=burn, chains=chains, ppc_samples=ppc_samples, export_long=export_long, model_code_sha256=u01_hddm_source_signature(), input_sha256={name: hashlib.sha256(u01_input_path(name).read_bytes()).hexdigest() for name in ['Participants.csv', 'Behavior_data.csv', 'HDDM_data.csv']})
        u01_atomic_record(out / 'fit_state.json', fit_state)
    long_path = out / 'tables/HDDM_posterior_draws.csv'
    wrote = False
    means = []
    diagrows = []
    for task in ['stroop', 'gonogo']:
        g = d[d.task.eq(task)].copy().reset_index(drop=True)
        expected = {'congruent', 'neutral', 'incongruent'} if task == 'stroop' else {'go', 'nogo'}
        if set(g.condition) != expected or set(g.subj_idx) != ids:
            raise ValueError('Missing task subjects or conditions')
        if not g.groupby('subj_idx').condition.apply(lambda x: set(x) == expected).all():
            raise ValueError('A subject lacks a model condition')
        if task == 'stroop':
            if not g.stimulus.isin([0, 1]).all() or g.rt.ge(998).any():
                raise ValueError('Stroop needs observed RT and actual color-boundary coding')
        elif (g.response.eq(0) != g.rt.eq(999)).any() or g.loc[g.response.eq(1), 'rt'].ge(998).any():
            raise ValueError('Go/No-Go lower boundary = nonresponse and RT 999; upper boundary = observed Go response')
        task_traces = []
        for chain in range(chains):
            u01_np.random.seed(seed + chain + (10000 if task == 'gonogo' else 0))
            kwargs = dict(depends_on={'v': 'condition'}, include=['a', 'v', 't', 'st'] + (['z'] if task == 'gonogo' else []), group_only_nodes=['st'], p_outlier=0.05)
            model = hddm.HDDMStimCoding(g, stim_col='stimulus', split_param='v', **kwargs) if task == 'stroop' else hddm.HDDM(g, **kwargs)
            model.find_starting_values()
            model.sample(samples, burn=burn, dbname=str(out / f'{task}_chain{chain}.db'), db='pickle')
            model.save(str(out / f'{task}_chain{chain}.model'))
            traces = model.get_traces()
            if traces.empty or not u01_np.isfinite(traces.to_numpy(float)).all():
                raise RuntimeError('Missing/nonfinite MCMC draws')
            traces.to_csv(out / f'{task}_chain{chain}_draws.csv', index=False)
            task_traces.append(traces)
            for node in traces:
                match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(node))
                if match:
                    (par, condition, sid) = match.groups()
                    parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
                    if sid not in ids or parameter not in u01_PARAMS:
                        raise ValueError('Unmapped subject parameter: ' + str(node))
                else:
                    (sid, parameter) = ('__group__', task + '_group_' + str(node))
                if export_long:
                    block = u01_pd.DataFrame(dict(task=task, chain=chain, draw=u01_np.arange(len(traces)), subject_id=sid, parameter=parameter, value=traces[node].to_numpy()))
                    block.to_csv(long_path, index=False, mode='a' if wrote else 'w', header=not wrote)
                    wrote = True
            if ppc_samples:
                ppc = u01_ddm_posterior_predictive_samples(model, task, ppc_samples, seed + chain)
                ppc.to_csv(out / f'{task}_chain{chain}_PPC.csv')
            del model
        if any((list(tr.columns) != list(task_traces[0].columns) or len(tr) != len(task_traces[0]) for tr in task_traces)):
            raise ValueError('MCMC chain node or draw mismatch')
        for node in task_traces[0]:
            match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(node))
            values = u01_np.stack([tr[node].to_numpy() for tr in task_traces])
            if match:
                (par, condition, sid) = match.groups()
                parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
                means.append(dict(subject_id=sid, parameter=parameter, value=values.mean()))
            if model_check:
                import arviz as az
                diagrows.append(dict(task=task, node=node, r_hat=float(az.rhat(values, method='rank')), ess_bulk=float(az.ess(values, method='bulk')), ess_tail=float(az.ess(values, method='tail'))))
    mean_table = u01_pd.DataFrame(means).pivot(index='subject_id', columns='parameter', values='value').reset_index()
    if set(u01_PARAMS) - set(mean_table) or len(mean_table) != len(ids):
        raise ValueError('Incomplete subject posterior means')
    mean_table['data_status'] = 'MODEL_CHECK_SIMULATION_REFITTED_NOT_EMPIRICAL' if model_check else 'FITTED_FROM_SUPPLIED_OBSERVATIONS'
    mean_table.to_csv(out / 'tables/HDDM_parameters.csv', index=False)
    if model_check:
        diagnostics = u01_pd.DataFrame(diagrows)
        diagnostics['diagnostics_pass'] = diagnostics.r_hat.lt(1.01) & diagnostics.ess_bulk.gt(400) & diagnostics.ess_tail.gt(400)
        diagnostics.to_csv(out / 'tables/Recovery_fit_diagnostics.csv', index=False)
        (out / 'MODEL_CHECK_ONLY.json').write_text(u01_json.dumps(dict(status='PARAMETER_RECOVERY_SIMULATION_REFIT_NOT_EMPIRICAL', seed=seed, samples=samples, burn=burn, chains=chains)))
    else:
        u01_provenance('hddm_fit', ['Participants.csv', 'Behavior_data.csv', 'HDDM_data.csv'], settings=dict(samples=samples, burn=burn, chains=chains, ppc_samples=ppc_samples, seed=seed, p_outlier=0.05))
        fit_state['status'] = 'complete'
        u01_atomic_record(out / 'fit_state.json', fit_state)
    return out


In [ ]:
def u01_ddm_validate_hddm_backend(hddm):
    import inspect as inspect
    from scipy import stats as stats
    if str(hddm.__version__) != '1.0.1':
        raise RuntimeError('Manuscript requires HDDM 1.0.1; install and validate the specified environment')
    source = inspect.getsource(hddm.likelihoods.generate_wfpt_stochastic_class)
    tree = __import__('ast').parse(source)
    fn = next((n for n in tree.body[0].body if isinstance(n, __import__('ast').FunctionDef) and n.name == 'wfpt_like'))
    namespace = dict(hddm=hddm, np=u01_np, stats=stats, wp=dict(err=0.0001, n_st=2, n_sz=2, use_adaptive=1, simps_err=0.001, w_outlier=0.1))
    module = __import__('ast').Module(body=[fn], type_ignores=[])
    exec(compile(module, 'installed_hddm_missing_rt_check', 'exec'), namespace)
    for v in [-1.0, 0.0, 1.0]:
        expected = 0.4 if v == 0 else u01_np.expm1(-2 * 1.2 * 0.4 * v) / u01_np.expm1(-2 * 1.2 * v)
        for boundary in [-1, 1]:
            val = namespace['wfpt_like'](u01_pd.DataFrame({'rt': [boundary * 999.0]}), v, 0, 1.2, 0.4, 0, 0.2, 0.05, p_outlier=0.05)
            prob = expected if boundary == 1 else 1 - expected
            if not u01_np.isclose(val, u01_np.log(prob), rtol=1e-06, atol=1e-08):
                raise RuntimeError('Installed HDDM does not implement the documented implicit-boundary missing-RT likelihood')
    return True


In [ ]:
def u01_ddm_recovery_sample(model, task, position):
    from kabuki.analyze import _parents_to_random_posterior_sample as _parents_to_random_posterior_sample
    traces = model.get_traces()
    if not 0 <= position < len(traces):
        raise ValueError('Posterior draw index outside fitted chain')
    truth = []
    for (name, value) in traces.iloc[position].items():
        match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(name))
        if match:
            (par, condition, sid) = match.groups()
            parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
            truth.append(dict(subject_id=sid, parameter=parameter, generating_value=float(value)))
    generated = []
    for (name, record) in model.iter_observeds():
        node = record['node']
        _parents_to_random_posterior_sample(node, pos=position)
        sample = node.random(add_outliers=True)
        observed = model.data.loc[node.value.index].copy()
        if len(sample) != len(observed):
            raise ValueError('Recovery generator changed trial counts')
        signed_rt = sample['rt'].to_numpy(float)
        observed['response'] = (signed_rt > 0).astype(int)
        observed['rt'] = u01_np.abs(signed_rt)
        if task == 'gonogo':
            observed.loc[observed.response.eq(0), 'rt'] = 999.0
        observed['task'] = task
        generated.append(observed)
    data = u01_pd.concat(generated).sort_index().reset_index(drop=True)
    if len(data) != len(model.data):
        raise ValueError('Recovery generation did not cover all observed trials')
    if len(truth) != len(u01_participants()) * (5 if task == 'stroop' else 5):
        raise ValueError('Incomplete generating subject parameters')
    return (data, u01_pd.DataFrame(truth))


In [ ]:
def u01_ddm_posterior_predictive_samples(model, task, samples=100, seed=20261021):
    rng = u01_np.random.default_rng(seed)
    rows = []
    original = model.data.sort_index().reset_index(drop=True)
    for sample in range(samples):
        position = int(rng.integers(len(model.get_traces())))
        (generated, _) = u01_ddm_recovery_sample(model, task, position)
        if not generated[['subj_idx', 'condition', 'stimulus']].equals(original[['subj_idx', 'condition', 'stimulus']]):
            raise ValueError('Posterior prediction lost original trial alignment')
        block = original.copy()
        block['rt_sampled'] = u01_np.where(generated.response.eq(1), generated.rt, -generated.rt)
        block['response_sampled'] = generated.response.to_numpy()
        block['data_status'] = 'POSTERIOR_PREDICTIVE_SIMULATION_WITH_APPENDED_OBSERVATIONS'
        block['sample'] = sample
        block['posterior_draw'] = position
        rows.append(block)
    return u01_pd.concat(rows, ignore_index=True)


In [ ]:
from pathlib import Path as u05_Path
import sys as u05_sys
from IPython.display import display as u05_display, Image as u05_Image
import numpy as u05_np
import pandas as u05_pd
u05_analysis_data = u00_behavior_analysis_data
u05_computational_COV = ['age', 'sex', 'education']
u05_computational_PAIRS = [('v', 'stroop_v_congruent', 'gonogo_v_go'), ('a', 'stroop_a', 'gonogo_a'), ('t', 'stroop_t', 'gonogo_t'), ('specific', 'stroop_specific', 'gonogo_specific')]


In [ ]:
def u05_computational_save(d, name):
    (u05_computational_OUT / 'tables').mkdir(parents=True, exist_ok=True)
    d.to_csv(u05_computational_OUT / 'tables' / name, index=False)
    return d


In [ ]:
def u05_computational_matched_contrasts(bootstrap=5000, seed=20261021):
    d = u05_analysis_data()
    specs = [('stroop_rt_cost_ms', 'v_stroop_conflict', 'v_gonogo'), ('gonogo_dprime', 'v_gonogo', 'v_stroop_conflict'), ('stroop_mean_rt_ms', 't_stroop', 't_gonogo'), ('gonogo_go_rt_ms', 't_gonogo', 't_stroop')]
    rows = []
    for (y, matched, cross) in specs:

        def estimate(b):
            X = u05_np.c_[u05_np.ones(len(b)), u05_standardized(b[u05_computational_COV + [matched, cross]])]
            beta = u05_np.linalg.lstsq(X, u05_standardized(b[y]), rcond=None)[0]
            return beta[-2:]
        obs = estimate(d)
        rng = u05_np.random.default_rng(seed)
        vals = []
        for _ in range(bootstrap):
            b = d.iloc[rng.integers(0, len(d), len(d))]
            beta = estimate(b)
            vals.append(beta[0] - beta[1])
        ci = u05_np.quantile(vals, [0.025, 0.975])
        rows.append(dict(outcome=y, matched=matched, cross=cross, beta_matched=obs[0], beta_cross=obs[1], delta_beta=obs[0] - obs[1], CI95_low=ci[0], CI95_high=ci[1], p=u05_bootstrap_p(vals), bootstrap_draws=bootstrap))
    out = u05_pd.DataFrame(rows)
    out['q_FDR_4'] = u05_fdr(out.p)
    u05_computational_save(out, 'Matched_minus_cross_parameter_beta.csv')
    u05_computational_forest(out, 'Matched_minus_cross_parameter_beta.png', 'outcome', 'delta_beta', 'CI95_low', 'CI95_high')
    return out


In [ ]:
def u05_computational_forest(table, filename, label, estimate, low, high):
    import matplotlib.pyplot as plt
    (fig, ax) = plt.subplots(figsize=(3.15, 2.0))
    for (i, row) in table.reset_index(drop=True).iterrows():
        ax.plot([row[low], row[high]], [i, i], color='#3C5488', lw=1.1)
        ax.scatter(row[estimate], i, color='#3C5488', s=20)
    ax.tick_params(labelsize=7)
    fig.tight_layout(pad=0.4)
    ax.axvline(0, color='0.6', lw=0.7)
    ax.set(yticks=range(len(table)), yticklabels=table[label], xlabel=estimate + ' (95% subject bootstrap CI)')
    (u05_computational_OUT / 'image').mkdir(parents=True, exist_ok=True)
    fig.savefig(u05_computational_OUT / 'image' / filename, dpi=600, bbox_inches='tight')
    plt.close(fig)


In [ ]:
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
from importlib import metadata as u15_metadata
import platform as u15_platform, sys as u15_sys
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_match_cohort(d, name, keys=('subject_id',), complete=True):
    ids = set(u15_participants().subject_id)
    if d.subject_id.isna().any() or not set(d.subject_id) <= ids:
        raise ValueError(f'{name}: unexpected or missing subject_id')
    if complete and set(d.subject_id) != ids:
        raise ValueError(f'{name}: cohort mismatch; do not fill missing participants')
    if d.duplicated(list(keys)).any():
        raise ValueError(f'{name}: duplicate {keys}')
    return d


In [ ]:
def u15_behavior():
    d = u15_table('Behavior_data.csv', ['subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'], ['subject_id', 'task', 'run', 'trial_id'])
    u15_match_cohort(d, 'Behavior_data', ('subject_id', 'task', 'run', 'trial_id'))
    if not d.task.isin(['stroop', 'gonogo']).all():
        raise ValueError('task must be stroop/gonogo')
    d['rt_s'] = u15_pd.to_numeric(d.rt_s, errors='raise')
    d['response'] = u15_pd.to_numeric(d.response, errors='raise')
    d['correct'] = u15_pd.to_numeric(d.correct, errors='raise')
    if not d.correct.isin([0, 1]).all() or not d.response.dropna().isin([0, 1]).all():
        raise ValueError('correct and observed response must be binary')
    if not u15_np.isfinite(d.rt_s.dropna()).all() or (d.rt_s.dropna() <= 0).any():
        raise ValueError('RTs must be in seconds; nonresponses have NaN RT')
    for (task, conds) in [('stroop', {'congruent', 'neutral', 'incongruent'}), ('gonogo', {'go', 'nogo'})]:
        g = d[d.task.eq(task)]
        if set(g.subject_id) != set(u15_participants().subject_id) or set(g.condition) != conds:
            raise ValueError(f'{task}: incomplete subjects/conditions')
        if not g.groupby('subject_id').condition.apply(lambda x: set(x) == conds).all():
            raise ValueError(f'{task}: a participant is missing a condition')
    return d


In [ ]:
def u15_posterior():
    d = u15_table('HDDM_parameters.csv', ['subject_id'] + u15_PARAMS, ['subject_id'])
    u15_match_cohort(d, 'HDDM_parameters')
    if not u15_np.isfinite(d[u15_PARAMS].to_numpy(float)).all():
        raise ValueError('Nonfinite fitted parameter')
    for c in ['stroop_a', 'gonogo_a', 'stroop_t', 'gonogo_t']:
        if (d[c] <= 0).any():
            raise ValueError(f'Parameter outside domain: {c}')
    if not d.gonogo_z.between(0, 1, inclusive='neither').all():
        raise ValueError('gonogo_z outside (0,1)')
    u15_verify_fitted_artifact(u15_input_path('HDDM_parameters.csv'))
    u15_provenance('posterior_input', ['HDDM_parameters.csv', 'Participants.csv'])
    return d


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
def u15_runtime_versions():
    versions = {'python': u15_sys.version, 'platform': u15_platform.platform()}
    for name in ['numpy', 'scipy', 'pandas', 'statsmodels', 'scikit-learn', 'nilearn', 'nibabel', 'arviz', 'hddm', 'kabuki', 'pymc', 'matplotlib', 'openpyxl', 'antspyx', 'templateflow']:
        try:
            versions[name] = u15_metadata.version(name)
        except u15_metadata.PackageNotFoundError:
            versions[name] = None
    return versions


In [ ]:
def u15_atomic_record(path, record):
    path = u15_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.pending')
    temp.write_text(u15_json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


In [ ]:
def u15_provenance(stage, names, settings=None):
    records = []
    for name in names:
        path = u15_input_path(name)
        records.append({'file': str(path), 'sha256': u15_hashlib.sha256(path.read_bytes()).hexdigest()})
    record = dict(stage=stage, input_files=records, code_sha256=u15_source_signature(), runtime=u15_runtime_versions(), settings=settings or {})
    u15_atomic_record(u15_RESULTS / (stage + '_inputs.json'), record)
    return record


In [ ]:
def u15_verify_fitted_artifact(path):
    path = u15_Path(path).resolve()
    generated = (u15_RESULTS / '2.hddm_fitting/tables').resolve()
    if path.parent != generated:
        return
    state_path = generated.parent / 'fit_state.json'
    if not state_path.is_file():
        raise RuntimeError('Generated HDDM tables lack a completed fit record; rerun the fit or use explicitly supplied external estimates')
    state = u15_json.loads(state_path.read_text())
    if state.get('model_code_sha256') != u15_hddm_source_signature():
        raise RuntimeError('HDDM model implementation changed since fitting; refit before using generated estimates')
    if state.get('status') != 'complete':
        raise RuntimeError('HDDM fit is incomplete; downstream analyses cannot read its tables')
    for (name, expected) in state['input_sha256'].items():
        if u15_hashlib.sha256(u15_input_path(name).read_bytes()).hexdigest() != expected:
            raise RuntimeError('Observed inputs changed since HDDM fitting: ' + name)
    diagnostics = generated / 'HDDM_sampling_diagnostics.csv'
    if path.name == 'HDDM_parameters.csv':
        if not diagnostics.is_file() or diagnostics.stat().st_mtime_ns < state_path.stat().st_mtime_ns:
            raise RuntimeError('Run chain diagnostics for the completed fit before associations')
        check = u15_pd.read_csv(diagnostics, dtype={'subject_id': str})
        if check.empty or not check.diagnostics_pass.astype(str).str.lower().eq('true').all():
            raise RuntimeError('HDDM sampling diagnostics did not all pass')


In [ ]:
def u15_observed_hddm_data():
    d = u15_table('HDDM_data.csv', unique=['subj_idx', 'task', 'run', 'trial_id']).rename(columns={'subj_idx': 'subject_id'})
    raw = u15_behavior()
    keep = raw.task.eq('gonogo') | raw.task.eq('stroop') & raw.rt_s.notna() & raw.response.notna()
    raw = raw.loc[keep]
    keys = ['subject_id', 'task', 'run', 'trial_id']
    if set(map(tuple, d[keys].to_numpy())) != set(map(tuple, raw[keys].to_numpy())):
        raise ValueError('HDDM and behavioral trial sets differ; retain observed Stroop responses and every Go/No-Go trial')
    joined = d.merge(raw[keys + ['condition', 'rt_s', 'response', 'correct']], on=keys, suffixes=('', '_behavior'), validate='one_to_one')
    if not joined.condition.eq(joined.condition_behavior).all():
        raise ValueError('HDDM and behavioral condition coding differs')
    if not joined.response.isin([0, 1]).all() or not u15_np.isfinite(joined.rt).all():
        raise ValueError('Invalid HDDM response or RT')
    st = joined.task.eq('stroop')
    if not joined.loc[st, 'stimulus'].isin([0, 1]).all() or not joined.loc[st, 'response'].eq(joined.loc[st, 'response_behavior']).all():
        raise ValueError('Stroop stimulus/response coding mismatch')
    if not joined.loc[st, 'correct'].eq(joined.loc[st, 'response'].eq(joined.loc[st, 'stimulus']).astype(int)).all():
        raise ValueError('Stroop correctness differs from the color-boundary coding')
    ng = ~st
    observed = joined.response.eq(1) | st
    if not u15_np.allclose(joined.loc[observed, 'rt'], joined.loc[observed, 'rt_s'], rtol=1e-08, atol=1e-09):
        raise ValueError('HDDM RTs differ from observed behavioral RTs')
    if not joined.loc[ng, 'response'].eq(joined.loc[ng, 'response_behavior']).all():
        raise ValueError('Go/No-Go response coding mismatch')
    if not joined.loc[ng & ~observed, 'rt'].eq(999).all() or joined.loc[ng & ~observed, 'rt_s'].notna().any():
        raise ValueError('Go/No-Go nonresponses must have blank behavioral RT and HDDM RT 999')
    return d.rename(columns={'subject_id': 'subj_idx'})


In [ ]:
def u15_hddm_source_signature():
    import types as types
    fit = u01_ddm_fit_hddm
    backend = u01_ddm_validate_hddm_backend
    sample = u01_ddm_recovery_sample
    predictive = u01_ddm_posterior_predictive_samples
    import re as re

    def normalized(code):
        return [code.co_code.hex(), [normalized(x) if isinstance(x, types.CodeType) else repr(x) for x in code.co_consts], [re.sub('^u[0-9]+_', '', x) for x in code.co_names], list(code.co_varnames), code.co_argcount, code.co_kwonlyargcount]
    payload = [normalized(f.__code__) for f in [fit, backend, sample, predictive, u15_observed_hddm_data]]
    return u15_hashlib.sha256(u15_json.dumps(payload, sort_keys=True).encode()).hexdigest()


In [ ]:
import numpy as u16_np
from scipy import stats as u16_stats, integrate as u16_integrate
from statsmodels.stats.multitest import multipletests as u16_multipletests


In [ ]:
def u16_fdr(p):
    a = u16_np.asarray(p, float)
    out = u16_np.full(a.shape, u16_np.nan)
    good = u16_np.isfinite(a)
    if good.any():
        out[good] = u16_multipletests(a[good], method='fdr_bh')[1]
    return out


In [ ]:
def u16_standardized(x):
    a = u16_np.asarray(x, float)
    if not u16_np.isfinite(a).all():
        raise ValueError('Nonfinite values cannot be standardized')
    sd = a.std(axis=0, ddof=1)
    if u16_np.any(sd <= 1e-12):
        raise ValueError('Constant variable cannot be standardized')
    return (a - a.mean(axis=0)) / sd


In [ ]:
def u16_partial_stat(x, y, cov):
    (x, y, cov) = (u16_np.asarray(x, float), u16_np.asarray(y, float), u16_np.asarray(cov, float))
    C = u16_np.c_[u16_np.ones(len(x)), cov]
    k = u16_np.linalg.matrix_rank(C) - 1
    if len(x) <= k + 3 or not u16_np.isfinite(u16_np.c_[x, y, C]).all():
        raise ValueError('Insufficient finite observations for partial correlation')
    e = u16_np.c_[x, y] - C @ u16_np.linalg.lstsq(C, u16_np.c_[x, y], rcond=None)[0]
    if u16_np.any(e.std(0) <= 1e-12):
        raise ValueError('Zero residual variance')
    r = float(u16_np.clip(u16_np.corrcoef(e.T)[0, 1], -1, 1))
    df = len(x) - k - 2
    t = r * u16_np.sqrt(df / max(1 - r * r, 1e-15))
    return dict(n=len(x), df=df, r=r, p=2 * u16_stats.t.sf(abs(t), df), nuisance_rank=k)


In [ ]:
def u16_equivalence(r, n, k, bound=0.3):
    if not 0 < bound < 1 or n <= k + 3:
        raise ValueError('Invalid equivalence specification')
    zz = u16_np.arctanh(u16_np.clip(r, -1 + 1e-12, 1 - 1e-12))
    se = 1 / u16_np.sqrt(n - k - 3)
    limit = u16_np.arctanh(bound)
    ci = u16_np.tanh(zz + u16_np.array([-1, 1]) * u16_stats.norm.ppf(0.95) * se)
    return dict(bound_r=bound, r90_low=ci[0], r90_high=ci[1], p_TOST=max(u16_stats.norm.sf((zz + limit) / se), u16_stats.norm.cdf((zz - limit) / se)))


In [ ]:
def u16_jzs_nested(y, X_full, X_null):
    y = u16_np.asarray(y, float)
    n = len(y)
    sst = u16_np.sum((y - y.mean()) ** 2)
    if sst <= 0:
        raise ValueError('Constant outcome')
    designs = [u16_np.c_[u16_np.ones(n), u16_np.asarray(v, float)] for v in [X_full, X_null]]
    if any((u16_np.linalg.matrix_rank(X) != X.shape[1] for X in designs)):
        raise ValueError('JZS regression requires full-rank designs')
    if not u16_np.allclose(designs[1], designs[0] @ u16_np.linalg.lstsq(designs[0], designs[1], rcond=None)[0]):
        raise ValueError('Null model is not nested in alternative')
    values = []
    for X in designs:
        p = X.shape[1] - 1
        if n <= p + 2:
            raise ValueError('Insufficient observations for JZS regression')
        err = y - X @ u16_np.linalg.lstsq(X, y, rcond=None)[0]
        r2 = float(u16_np.clip(1 - u16_np.sum(err ** 2) / sst, 0, 1 - 1e-14))

        def logfun(u):
            return (n - 1 - p) / 2 * u16_np.logaddexp(0, u) - (n - 1) / 2 * u16_np.logaddexp(0, u16_np.log1p(-r2) + u) - 0.5 * u - n / 2 * u16_np.exp(-u)
        grid = u16_np.linspace(-25, 60, 400)
        shift = max((logfun(u) for u in grid))
        integral = u16_integrate.quad(lambda u: u16_np.exp(logfun(u) - shift), -30, 80, epsabs=1e-10, epsrel=1e-09, limit=300)[0]
        if integral <= 0 or not u16_np.isfinite(integral):
            raise ArithmeticError('JZS integration failed')
        values.append(u16_np.log(integral) + shift)
    bf = float(u16_np.exp(u16_np.clip(values[0] - values[1], -700, 700)))
    return dict(BF10=bf, BF01=1 / bf, JZS_prior='g ~ inverse-gamma(1/2, n/2); Wetzels & Wagenmakers (2012), Eq. 15')


In [ ]:
def u16_bootstrap_p(draws):
    d = u16_np.asarray(draws, float)
    if len(d) == 0 or not u16_np.isfinite(d).all():
        raise ValueError('Bootstrap evidence requires finite nonempty draws')
    return u16_np.minimum(1, 2 * u16_np.minimum((u16_np.sum(d <= 0, axis=0) + 1) / (len(d) + 1), (u16_np.sum(d >= 0, axis=0) + 1) / (len(d) + 1)))


In [ ]:
u00_participants = u15_participants
u00_behavior = u15_behavior
u00_posterior = u15_posterior
u01_RESULTS = u15_RESULTS
u01_hddm_source_signature = u15_hddm_source_signature
u01_RESULTS = u15_RESULTS
u01_participants = u15_participants
u01_input_path = u15_input_path
u01_provenance = u15_provenance
u01_PARAMS = u15_PARAMS
u01_observed_hddm_data = u15_observed_hddm_data
u01_atomic_record = u15_atomic_record
u01_ddm_OUT = u01_RESULTS / '2.hddm_fitting'
u05_ROOT = u15_ROOT
u05_DATA = u15_DATA
u05_RESULTS = u15_RESULTS
u05_RESULTS = u15_RESULTS
u05_provenance = u15_provenance
u05_standardized = u16_standardized
u05_partial_stat = u16_partial_stat
u05_equivalence = u16_equivalence
u05_jzs_nested = u16_jzs_nested
u05_fdr = u16_fdr
u05_bootstrap_p = u16_bootstrap_p
u05_computational_OUT = u05_RESULTS / '3.parameter_behavior'


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u05_display = a8_display_three_line


In [ ]:
u05_matched_results = u05_computational_matched_contrasts(bootstrap=5000)
u05_display(u05_matched_results)
